# Notebook: 00 Image Extract
### Purpose: unzip archives, strip `__MACOSX`, build train masks from annotations.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
# ponytail: PROJECT_ROOT or parent of notebooks/
root = Path(os.getenv("PROJECT_ROOT") or (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)).resolve()
sys.path[:0] = [str(root), str(root / "src")]

import random
import shutil
import zipfile

from src.data.annotations import load_json_annotations
from src.data.masks import save_mask
from src.utils.config import Config
from src.utils.helpers import init_notebook, p, t

config = Config.load(root=root)
init_notebook(config.train.seed)

train_zip = config.paths.train_images_zip
eval_zip = config.paths.eval_images_zip
train_dir = config.paths.train_images
eval_dir = config.paths.eval_images
mask_dir = config.paths.train_masks


#### Extract images and remove `__MACOSX` folders


In [ ]:
# Mapping of zip files to their extraction targets
extraction_map = [
    (train_zip, train_dir),
    (eval_zip, eval_dir),
]

for zip_path, extract_to in extraction_map:
    if zip_path and Path(zip_path).exists():

        p("Working", str(zip_path))

        extract_to.mkdir(parents = True, exist_ok = True)

        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_to)

            src_rel = Path(zip_path).resolve().relative_to(root)
            dst_rel = Path(extract_to).resolve().relative_to(root)
            p("Extracted", f"{src_rel} -> {dst_rel}")

        for m in extract_to.rglob("__MACOSX"):
            if m.is_dir():
                shutil.rmtree(m)
                rel = m.resolve().relative_to(root)
                p("Removed", str(rel))

        p()
    elif zip_path:
        p("Zip path not found", str(zip_path))


#### Build masks from annotations


In [ ]:
annotations_path = config.paths.annotations
entries = load_json_annotations(annotations_path)
mask_dir.mkdir(parents = True, exist_ok = True)

for entry in entries:
    image_path = train_dir / entry.image_path.name
    if not image_path.exists():
        continue

    mask = entry.to_mask()
    save_path = mask_dir / entry.image_path.name
    save_mask(mask, save_path)
